# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Anusha-hasan25/flyrank-ml-AnushaHasan/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [3]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df["trend_direction"].value_counts())


trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Anusha-hasan25/flyrank-ml-AnushaHasan"
REPO_DIR = "flyrank-ml-AnushaHasan"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")  # from work/notebooks/ back to repo root

print("Working dir:", os.getcwd())
print("Data file exists:", os.path.exists("data/raw/content_refresh_anonymized.csv"))

Working dir: /content/flyrank-ml-AnushaHasan
Data file exists: True


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(df["is_declining_label"].value_counts(normalize=True).round(3))

is_declining_label
1    0.542
0    0.458
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
def precision_at_k(scores, labels, k):
    import numpy as np
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()
# reference number from your own w02 hand-rule run:
print("Example hand-rule Precision@50 from earlier notebook: ~0.24-0.34 depending on rule/version")


Example hand-rule Precision@50 from earlier notebook: ~0.24-0.34 depending on rule/version


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df[["content_id", "client_id", "impressions_90d", "days_since_last_update",
    "avg_position", "ctr", "word_count", "trend_direction", "is_declining_label"]].head(10)

,content_id,client_id,impressions_90d,days_since_last_update,avg_position,ctr,word_count,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,20,10.6,0.76,3221.0,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,25,20.3,0.05,2481.0,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,20,36.5,0.09,3515.0,down,1
3,content_331d6c4de07b,client_19581e27de,11751,22,6.2,0.49,NaN,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,14,44.0,0.13,2803.0,down,1
5,content_d4084a4bc775,client_f369cb89fc,3970,20,8.5,0.03,3080.0,down,1
6,content_9a34b442b552,client_8722616204,20,20,7.0,0.00,3059.0,down,1
7,content_a63219c6e95a,client_19581e27de,1724,22,21.2,0.06,NaN,stable,0
8,content_5e6c160719bc,client_6208ef0f77,32574,20,46.0,0.09,3807.0,down,1
9,content_c27558df2b0c,client_19581e27de,1240,104,4.9,0.16,NaN,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.tree import DecisionTreeClassifier, export_text
features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].fillna(0)
y = df["is_declining_label"]
tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X, y)
print(export_text(tree, feature_names=features))

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.